In [1]:
import sys, platform
print('Python 解释器路径:', sys.executable)
print('Python 版本:', sys.version)
print('平台:', platform.platform())
# 顺便打印依赖版本
import pandas, lightgbm
print('pandas:', pandas.__version__)
print('lightgbm:', lightgbm.__version__)


Python 解释器路径: /opt/homebrew/opt/python@3.11/bin/python3.11
Python 版本: 3.11.4 (main, Jun 20 2023, 17:37:48) [Clang 14.0.0 (clang-1400.0.29.202)]
平台: macOS-26.5.2-arm64-arm-64bit
pandas: 3.0.1
lightgbm: 4.7.0


## 1. 依赖与常量

In [2]:
"""不重叠采样 walk-forward + 止损（R 口径）：一次一仓 + 持仓 H 分钟 + 止损 N×ATR。"""
import argparse
import math
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

import sys
sys.path.insert(0, str(Path('.').resolve()))
from ml_orderflow_v2 import _agg_minute, build_features

MBODIR = Path(".") / "data_tick" / "mbo_days"
F = 0.01

## 2. 加载 mbo 数据（分钟级聚合，分天防 OOM）

In [3]:
def load_minute():
    frames = []
    for day_dir in sorted(MBODIR.iterdir()):
        if not day_dir.is_dir():
            continue
        files = sorted(day_dir.glob("*.parquet"))
        if not files:
            continue
        parts = [pd.read_parquet(f) for f in files]
        valid_parts = [p for p in parts if len(p)]
        if not valid_parts:
            continue
        frames.append(_agg_minute(pd.concat(valid_parts)))
        del parts, valid_parts
    return pd.concat(frames).sort_index()

min_df = load_minute()
print(f"分钟级: {len(min_df)}（{min_df.index.date.min()} ~ {min_df.index.date.max()}）")

分钟级: 25200（2026-06-29 ~ 2026-08-05）


In [4]:
min_df.head()

,ofi,n_buy,n_sell,avg_size,open,high,low,close,vol,flow_large
ts,,,,,,,,,,
2026-06-29 08:00:00+00:00,40975,1130,775,89.177428,714.63,750.00,684.00,714.68,169883,29245
2026-06-29 08:01:00+00:00,10335,951,920,107.292892,714.49,716.99,687.21,714.99,200745,750
2026-06-29 08:02:00+00:00,16642,1099,1041,105.134579,715.00,719.12,704.88,715.55,224988,1175
2026-06-29 08:03:00+00:00,51078,831,909,127.271264,715.35,716.99,708.00,714.54,221452,56215
2026-06-29 08:04:00+00:00,-24670,476,762,92.660743,714.78,715.18,708.40,714.92,114714,-13966


## 3. ATR 计算

In [3]:
def add_atr(min_df, period=14):
    prev_close = min_df["close"].shift(1)
    tr = pd.concat([
        min_df["high"] - min_df["low"],
        (min_df["high"] - prev_close).abs(),
        (min_df["low"] - prev_close).abs(),
    ], axis=1).max(axis=1)
    min_df["atr"] = tr.rolling(period).mean()
    return min_df

min_df = add_atr(min_df)
print("ATR 分布:", min_df["atr"].describe()[["mean","50%","min"]].to_dict())

ATR 分布: {'mean': 28.55732120878685, '50%': 12.83285714285714, 'min': 0.0}


## 4. 不重叠采样（label 窗口 H 分钟，每 H 分钟采一个）

In [4]:
def build_nonoap(min_df, H):
    feats = build_features(min_df, 60)
    close = min_df["close"]
    n = len(min_df)
    X_list, y_list, e_idx, x_idx = [], [], [], []
    for t in range(0, n - H, H):
        if feats.iloc[t].isna().all():
            continue
        fr = close.iloc[t + H] / close.iloc[t] - 1
        if not np.isfinite(fr):
            continue
        X_list.append(feats.iloc[t].values)
        y_list.append(1 if fr > 0 else 0)
        e_idx.append(t)
        x_idx.append(t + H)
    return np.array(X_list), np.array(y_list), np.array(e_idx), np.array(x_idx)

H = 6
X, y, e_idx, x_idx = build_nonoap(min_df, H)
print(f"不重叠样本: {len(X)}（H={H}）")

不重叠样本: 4177（H=6）


/var/folders/0n/sz6dbf6s3mx1f9jvzt0dc7km0000gn/T/ipykernel_43806/3775121660.py:9: RuntimeWarning: divide by zero encountered in scalar divide
  fr = close.iloc[t + H] / close.iloc[t] - 1
/var/folders/0n/sz6dbf6s3mx1f9jvzt0dc7km0000gn/T/ipykernel_43806/3775121660.py:9: RuntimeWarning: invalid value encountered in scalar divide
  fr = close.iloc[t + H] / close.iloc[t] - 1


## 5. 超参数设置

In [5]:
stop_atr = 2.0    # 止损 N×ATR（0=无止损）
train_days, test_days = 8, 2
n_est = 300

## 6. walk-forward 主循环（滚动 8 天训练 / 2 天测试）

In [6]:
close_all = min_df["close"].values
atr_all = min_df["atr"].values
high_all = min_df["high"].values
low_all = min_df["low"].values

spd = 390 // H
train_n, test_n = train_days * spd, test_days * spd
results = []
start = 0
while start + train_n + test_n <= len(X):
    tr = slice(start, start + train_n)
    te = slice(start + train_n, start + train_n + test_n)
    model = lgb.LGBMClassifier(n_estimators=n_est, learning_rate=0.05, num_leaves=63,
                               subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=2)
    model.fit(X[tr], y[tr])
    proba = model.predict_proba(X[te])[:, 1]
    auc = roc_auc_score(y[te], proba) if len(set(y[te])) > 1 else float("nan")
    q = np.array(pd.qcut(proba, 5, labels=False, duplicates="drop"))
    # 一次一仓 + 止损
    trades = []
    pos, entry_p, risk, stop_p, pos_end_min, t0 = 0, 0.0, 0.0, 0.0, -1, 0
    te_e, te_x = e_idx[te.start:te.stop], x_idx[te.start:te.stop]
    for i in range(len(te_e)):
        if pos == 0 and q[i] in (0, 4):
            pos = 1 if q[i] == 4 else -1
            t0 = te_e[i]
            entry_p = close_all[t0]
            risk = stop_atr * atr_all[t0] if stop_atr > 0 else 1.0
            stop_p = entry_p - pos * risk
            pos_end_min = te_x[i]
        elif pos != 0 and te_e[i] >= pos_end_min:
            exited = False
            for t in range(t0 + 1, pos_end_min):
                if pos > 0 and low_all[t] <= stop_p:
                    gross_r = (stop_p - entry_p) / risk
                    trades.append((gross_r, "stop", pos, entry_p, t))
                    exited = True; pos = 0; break
                elif pos < 0 and high_all[t] >= stop_p:
                    gross_r = (entry_p - stop_p) / risk
                    trades.append((gross_r, "stop", pos, entry_p, t))
                    exited = True; pos = 0; break
            if not exited:
                exit_p = close_all[pos_end_min]
                gross_r = pos * (exit_p - entry_p) / risk
                cost_r = 3e-4 * (entry_p + exit_p) / risk
                trades.append((gross_r - cost_r, "time", pos, entry_p, pos_end_min))
                pos = 0
    if trades:
        net = np.array([t[0] for t in trades if np.isfinite(t[0])])
        net_mean = net.mean() if len(net) else float("nan")
        n_stop = sum(1 for t in trades if t[1] == "stop")
    else:
        net_mean, n_stop = float("nan"), 0
    results.append((start // spd, auc, net_mean, len(trades), n_stop))
    print(f"窗口{len(results)}: day{start//spd}  AUC {auc:.3f}  净R {net_mean:+.3f}  交易{len(trades)} 止损{n_stop}", flush=True)
    start += test_n

[LightGBM] [Info] Number of positive: 264, number of negative: 256
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000311 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2425
[LightGBM] [Info] Number of data points in the train set: 520, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.507692 -> initscore=0.030772
[LightGBM] [Info] Start training from score 0.030772
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


[LightGBM] [Info] Number of positive: 261, number of negative: 259
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000051 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2435
[LightGBM] [Info] Number of data points in the train set: 520, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.501923 -> initscore=0.007692
[LightGBM] [Info] Start training from score 0.007692
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

[LightGBM] [Info] Number of positive: 263, number of negative: 257
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000058 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2427
[LightGBM] [Info] Number of data points in the train set: 520, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.505769 -> initscore=0.023078
[LightGBM] [Info] Start training from score 0.023078
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Info] Number of positive: 262, number of negative: 258
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000032 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2434
[LightGBM] [Info] Number of data points in the train set: 520, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503846 -> initscore=0.015385
[LightGBM] [Info] Start training from score 0.015385
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


...

[LightGBM] [Info] Number of positive: 267, number of negative: 253
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000057 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2424
[LightGBM] [Info] Number of data points in the train set: 520, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.513462 -> initscore=0.053859
[LightGBM] [Info] Start training from score 0.053859
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

## 7. 汇总统计

In [7]:
aucs = [r[1] for r in results if np.isfinite(r[1])]
nets = [r[2] for r in results if np.isfinite(r[2])]
print(f"\n=== walk-forward 汇总（{len(results)} 窗口，H={H}，止损 {stop_atr}×ATR）===")
print(f"  AUC: 均值 {np.mean(aucs):.3f} | 正窗口(>0.52) {sum(1 for a in aucs if a>0.52)}/{len(aucs)}")
if nets:
    print(f"  净R: 均值 {np.mean(nets):+.3f} | 正窗口 {sum(1 for s in nets if s>0)}/{len(nets)}")
    print(f"  {'✅ 稳健' if np.mean(nets) > 0 and sum(1 for s in nets if s>0) >= len(nets)*0.6 else '❌ 不稳定'}")


=== walk-forward 汇总（28 窗口，H=6，止损 2.0×ATR）===
  AUC: 均值 0.587 | 正窗口(>0.52) 25/28
  净R: 均值 -0.099 | 正窗口 1/28
  ❌ 不稳定
